# 🔬 HyperMix — Quickstart

**Open detection of engineered biosignatures in remote hyperspectral imagery.**

This notebook runs in your browser (Google Colab), no setup. It:
1. simulates a realistic hyperspectral scene with a faint engineered reporter,
2. runs the classical matched-filter detector,
3. shows how detection degrades as signal-to-noise drops, and
4. trains the physics-informed learned detector and compares them.

Repo: https://github.com/JVLegend/HyperMix · MIT licensed · funded by the Experiment Foundation.

In [ ]:
!pip install -q "git+https://github.com/JVLegend/HyperMix.git" matplotlib

## 1. Simulate a scene

Each pixel is a full spectrum. A few faint reporter patches are hidden in a natural background, then blurred and buried in noise.

In [ ]:
import matplotlib.pyplot as plt
from hypermix import simulate_scene, false_color

scene = simulate_scene(snr_db=8.0, seed=0)
print('cube shape (x, y, wavelengths):', scene.cube.shape)

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(false_color(scene.cube)); ax[0].set_title('Simulated cube (false color)')
ax[1].imshow(scene.detection_gt, cmap='magma'); ax[1].set_title('Ground truth: reporter present')
for a in ax: a.axis('off')
plt.show()

## 2. Detect it with the classical matched filter

The matched filter is the standard tool. It works when the background is simple, and struggles when it is not.

In [ ]:
from hypermix import spectral_matched_filter, roc_auc

score = spectral_matched_filter(scene.cube, scene.reporter)
print('Detection AUC:', round(roc_auc(score, scene.detection_gt), 3))

plt.figure(figsize=(4.5, 4))
plt.imshow(score, cmap='magma'); plt.title('Matched filter score'); plt.axis('off'); plt.show()

## 3. Watch detection collapse as SNR drops

Long-range biosensing lives at low SNR. This is exactly where the classical detector breaks down.

In [ ]:
snrs = [30, 20, 10, 5, 0]
aucs = []
for snr in snrs:
    s = simulate_scene(snr_db=snr, seed=0)
    aucs.append(roc_auc(spectral_matched_filter(s.cube, s.reporter), s.detection_gt))

for snr, a in zip(snrs, aucs):
    print(f'SNR {snr:>2} dB  ->  AUC {a:.3f}')

plt.figure(figsize=(5, 3.5))
plt.plot(snrs, aucs, 'o-', color='#b8972a')
plt.gca().invert_xaxis(); plt.xlabel('SNR (dB)'); plt.ylabel('Detection AUC')
plt.title('Matched filter degrades at low SNR'); plt.grid(alpha=0.3); plt.show()

## 4. The learned detector

HyperMix trains a small physics-informed network on simulated scenes, using each scene's own adaptive detectors plus spatial context. Because the features are scene-relative, it generalizes to backgrounds it never saw. (Training takes ~15 s on CPU.)

In [ ]:
!pip install -q torch

In [ ]:
import numpy as np
from hypermix import reporter_library, implant_target, simulate_scene, spectral_matched_filter, roc_auc
from hypermix.detector import SpectralDetector, make_training_set

target = reporter_library(60)['bacteriochlorophyll_a']
X, y = make_training_set(target, n_scenes=16, hw=64)
det = SpectralDetector(n_features=X.shape[1], seed=0).fit(X, y, epochs=25)

# evaluate on a held-out background at low SNR
bg = simulate_scene(height=64, width=64, n_bands=60, snr_db=40.0, reporter_max_abundance=0.0, seed=999).cube
rng = np.random.default_rng(7)
test, gt, _, tgt = implant_target(bg, rng, target=target, snr_db=5.0)

mf = spectral_matched_filter(test, tgt)
mean, std = det.score_map(test, tgt, mc=25)
print('Matched filter AUC :', round(roc_auc(mf, gt), 3))
print('Learned  detector AUC:', round(roc_auc(mean, gt), 3))

fig, ax = plt.subplots(1, 4, figsize=(15, 4))
ax[0].imshow(gt, cmap='magma'); ax[0].set_title('Ground truth')
ax[1].imshow(mf, cmap='magma'); ax[1].set_title('Matched filter')
ax[2].imshow(mean, cmap='magma'); ax[2].set_title('Learned detector')
ax[3].imshow(std, cmap='viridis'); ax[3].set_title('Uncertainty (MC-dropout)')
for a in ax: a.axis('off')
plt.show()

## Next

The same detector, trained only on simulation, beats the matched filter on a **real** AVIRIS cube (Indian Pines). See the full benchmark and the paper-grounded reporters (biliverdin IXα, bacteriochlorophyll a) in the [repository](https://github.com/JVLegend/HyperMix).

Thank you for supporting open science. 🧬